In [2]:
!pip install transformers datasets evaluate rouge_score accelerate -U

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 149.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 51.9 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=8ede2cd6ac3ce6562c43e7d08af86d1bd705fd830e97fd7cc19cd29c09143b6d
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.2
    Uninstalli

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Loading dataset for evaluation...


Generating train split: 0 examples [00:00, ? examples/s]

Loaded 2502 test examples.
--- Generating Predictions ---


Generating:   0%|          | 1/2502 [01:00<42:18:21, 60.90s/it]


KeyboardInterrupt: 

In [5]:
import torch
import numpy as np
import evaluate
import os
from transformers import (
    BertTokenizer,
    EncoderDecoderModel,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
from datasets import load_dataset

# --- CONFIGURATION ---
# Verify these paths match your Drive structure!
DATA_PATH = "/content/drive/MyDrive/mbert/final_data.ndjson"
DRIVE_SAVE_PATH = "/content/drive/MyDrive/mbert"
CHECKPOINT = "bert-base-multilingual-cased"

# Disable WandB logging to prevent login prompts
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"

In [6]:
from transformers import BertTokenizer, EncoderDecoderModel, GenerationConfig

CHECKPOINT = "bert-base-multilingual-cased"

print("⏳ Loading Model & Tokenizer...")
tokenizer = BertTokenizer.from_pretrained(CHECKPOINT)
model = EncoderDecoderModel.from_encoder_decoder_pretrained(CHECKPOINT, CHECKPOINT)

# --- THE FIX: Configure Tokenizer EOS ---
# BERT doesn't have a standard EOS, so we map it to SEP
tokenizer.bos_token = tokenizer.cls_token
tokenizer.eos_token = tokenizer.sep_token

# --- CRITICAL FIX 1: Set Main Config ---
model.config.decoder_start_token_id = tokenizer.cls_token_id
model.config.eos_token_id = tokenizer.sep_token_id
model.config.pad_token_id = tokenizer.pad_token_id
model.config.vocab_size = model.config.encoder.vocab_size

# --- CRITICAL FIX 2: Create & Set Generation Config ---
# This is what the Trainer actually looks at during evaluation!
model.generation_config = GenerationConfig(
    decoder_start_token_id=tokenizer.cls_token_id,
    eos_token_id=tokenizer.sep_token_id,
    pad_token_id=tokenizer.pad_token_id,
    max_length=64,
    min_length=5,
    no_repeat_ngram_size=3,
    early_stopping=True,
    length_penalty=2.0,
    num_beams=4,
    vocab_size=model.config.encoder.vocab_size
)

print(f"✅ Config Locked! Decoder Start Token ID: {model.generation_config.decoder_start_token_id}")

⏳ Loading Model & Tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Some weights of BertLMHeadModel were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['bert.encoder.layer.0.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.0.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.0.crossattention.output.dense.bias', 'bert.encoder.layer.0.crossattention.output.dense.weight', 'bert.encoder.layer.0.crossattention.self.key.bias', 'bert.encoder.layer.0.crossattention.self.key.weight', 'bert.encoder.layer.0.crossattention.self.query.bias', 'bert.encoder.layer.0.crossattention.self.query.weight', 'bert.encoder.layer.0.crossattention.self.value.bias', 'bert.encoder.layer.0.crossattention.self.value.weight', 'bert.encoder.layer.1.crossattention.output.LayerNorm.bias', 'bert.encoder.layer.1.crossattention.output.LayerNorm.weight', 'bert.encoder.layer.1.crossattention.output.dense.bias', 'bert.encoder.layer.1.crossattention.output.dense.weight', 'bert.encoder.layer.1.crossattention.self.key.bia

✅ Config Locked! Decoder Start Token ID: 101


In [7]:
print(f"Loading data from {DATA_PATH}...")

# 1. Load Dataset (Using 'json' because file is .ndjson)
dataset = load_dataset("json", data_files=DATA_PATH)

# 2. Split if 'test' doesn't exist
if "test" not in dataset:
    print("Splitting into Train/Test...")
    dataset = dataset["train"].train_test_split(test_size=0.1)

# 3. Preprocessing Function
def preprocess_function(examples):
    inputs = examples["text"]      # Article text
    targets = examples["headline"] # Target Headline

    # Tokenize inputs (Articles)
    model_inputs = tokenizer(
        inputs, max_length=512, truncation=True, padding="max_length"
    )

    # Tokenize targets (Headlines)
    labels = tokenizer(
        text_target=targets, max_length=64, truncation=True, padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 4. Apply Mapping
print("Tokenizing dataset...")
tokenized_dataset = dataset.map(preprocess_function, batched=True)
print("✅ Data Ready!")

Loading data from /content/drive/MyDrive/mbert/final_data.ndjson...


Generating train split: 0 examples [00:00, ? examples/s]

Splitting into Train/Test...
Tokenizing dataset...


Map:   0%|          | 0/22515 [00:00<?, ? examples/s]

Map:   0%|          | 0/2502 [00:00<?, ? examples/s]

✅ Data Ready!


In [6]:
# 1. Setup Metrics
import evaluate
rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    # Replace -100 in the labels as we can't decode them
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

# 2. Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir=DRIVE_SAVE_PATH,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    dataloader_num_workers=2,
    weight_decay=0.01,
    save_total_limit=2,
    num_train_epochs=3,
    predict_with_generate=True, # This triggers the generation config we just fixed
    fp16=True,
    logging_steps=10,
    report_to="none",
)

# 3. Initialize Trainer
trainer = Seq2SeqTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
)

# 4. Train
print("🚀 Starting mBERT Training on A100 (With Fix)...")
trainer.train()

# 5. Save
trainer.save_model(DRIVE_SAVE_PATH)
tokenizer.save_pretrained(DRIVE_SAVE_PATH)
print(f"✅ Saved to {DRIVE_SAVE_PATH}")

/tmp/ipython-input-403200718.py:35: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 101}.


🚀 Starting mBERT Training on A100 (With Fix)...


We strongly recommend passing in an `attention_mask` since your input_ids may be padded. See https://huggingface.co/docs/transformers/troubleshooting#incorrect-output-when-padding-tokens-arent-masked.
/usr/local/lib/python3.12/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:575: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,1.332100,1.299351,0.828700,0.000000,0.828700,0.831300
2,1.170200,1.135229,1.087800,0.040000,1.084500,1.091100
3,1.096300,1.079651,1.672000,0.040000,1.668000,1.679300


/usr/local/lib/python3.12/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:575: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no need to pass them yourself anymore.
  warnings.warn(DEPRECATION_WARNING, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/models/encoder_decoder/modeling_encoder_decoder.py:575: FutureWarning: Version v4.12.0 introduces a better way to train encoder-decoder models by computing the loss inside the encoder-decoder framework rather than in the decoder itself. You may observe training discrepancies if fine-tuning a model trained with versions anterior to 4.12.0. The decoder_input_ids are now created based on the labels, no 

✅ Saved to /content/drive/MyDrive/mbert


In [8]:
import torch
import numpy as np
import evaluate
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Or EncoderDecoderModel if using Bert2Bert directly
from tqdm import tqdm

# ==========================================
# 1. Setup & Load Model from Drive
# ==========================================

# Mount Google Drive
drive.mount('/content/drive')

# Define the path where you saved the model
model_path = "/content/drive/MyDrive/mbert"

print(f"Loading model and tokenizer from: {model_path}")

# Load Tokenizer and Model
# Note: Ensure you use the same class used for training (e.g., AutoModelForSeq2SeqLM or EncoderDecoderModel)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print("Model loaded successfully!")

# ==========================================
# 2. Generate Predictions (The "Pick Up" Step)
# ==========================================

# ⚠️ ASSUMPTION: You have a list of source texts named 'test_texts'
# and a list of actual headlines named 'references'
# Example placeholder:
# test_texts = ["Your input article text here...", "Another article..."]
# references = ["The actual headline...", "Another headline..."]

print("--- Generating Predictions ---")
predictions = []

# Generation Loop
for text in tqdm(test_texts, desc="Generating"):
    # Tokenize input
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        padding="max_length"
    ).to(device)

    # Generate output
    # Adjust max_length, num_beams, etc., based on your training config
    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    # Decode output
    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(pred)

# ==========================================
# 3. Calculate Metrics (Your Code)
# ==========================================

print("--- Calculating Metrics ---")

# Load ROUGE metric
rouge = evaluate.load("rouge")

# 1. ROUGE Scores
rouge_results = rouge.compute(predictions=predictions, references=references)

# 2. BERTScore (Urdu)
# lang="ur" uses a multilingual model best suited for Urdu
bertscore = evaluate.load("bertscore")
bert_results = bertscore.compute(predictions=predictions, references=references, lang="ur")

# Extract averages
bert_precision = np.mean(bert_results['precision'])
bert_recall    = np.mean(bert_results['recall'])
bert_f1        = np.mean(bert_results['f1'])

# 3. Print Final Report
print("\n" + "="*30)
print("      mBERT Final Results")
print("="*30)
print(f"ROUGE-1:        {rouge_results['rouge1']*100:.2f}%")
print(f"ROUGE-2:        {rouge_results['rouge2']*100:.2f}%")
print(f"ROUGE-L:        {rouge_results['rougeL']*100:.2f}%")
print("-" * 30)
print(f"BERT Precision: {bert_precision*100:.2f}%")
print(f"BERT Recall:    {bert_recall*100:.2f}%")
print(f"BERT F1 Mean:   {bert_f1*100:.2f}%")
print("="*30)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model and tokenizer from: /content/drive/MyDrive/mbert


The tokenizer you are loading from '/content/drive/MyDrive/mbert' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Model loaded successfully!
--- Generating Predictions ---


NameError: name 'test_texts' is not defined

In [9]:
pip install bert_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.8 MB/s eta 0:00:00


In [16]:
# --- 1. Install Dependencies ---
!pip install transformers datasets evaluate rouge_score sacrebleu bert_score accelerate -U -q

# --- 2. Imports & Setup ---
import torch
import evaluate
import numpy as np
import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import load_dataset
from tqdm import tqdm

# --- 3. Mount Drive & Set Paths ---
drive.mount('/content/drive')
MODEL_PATH = "/content/drive/MyDrive/mbert"
DATA_PATH = "/content/drive/MyDrive/mbert/final_data.ndjson"

# --- 4. Load Model & Tokenizer ---
print(f"⏳ Loading model from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)
print(f"✅ Model loaded on {device}")

# --- 5. Load Data & Create Subset ---
print("⏳ Loading dataset...")
dataset = load_dataset("json", data_files=DATA_PATH)

# Ensure test split exists
if "test" not in dataset:
    dataset = dataset["train"].train_test_split(test_size=0.1, seed=42)

# Select only 20 samples
SUBSET_SIZE = 20
print(f"✂️ Slicing first {SUBSET_SIZE} samples for evaluation...")
test_texts = dataset["test"]["text"][:SUBSET_SIZE]
references = dataset["test"]["headline"][:SUBSET_SIZE]

# --- 6. Generate Predictions ---
print("--- Generating Predictions ---")
predictions = []

for text in tqdm(test_texts, desc="Generating"):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=512,
        truncation=True,
        padding="max_length"
    ).to(device)

    output_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    pred = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(pred)

# --- 7. Calculate Metrics ---
print("--- Calculating Metrics ---")
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
bertscore = evaluate.load("bertscore")

# --- THE FIX: Define helper to use the model's tokenizer ---
def tokenize_for_rouge(text):
    return tokenizer.tokenize(text)

# ROUGE (Now using the custom tokenizer)
rouge_results = rouge.compute(
    predictions=predictions,
    references=references,
    tokenizer=tokenize_for_rouge
)

# BLEU (Requires list of lists for references)
references_for_bleu = [[ref] for ref in references]
bleu_results = bleu.compute(predictions=predictions, references=references_for_bleu)

# BERTScore (Urdu)
bert_results = bertscore.compute(predictions=predictions, references=references, lang="ur")

# --- 8. Print Results ---
# (Multiplied by 100 to show actual percentages)
print("\n" + "="*40)
print(f"      FINAL RESULTS (N={SUBSET_SIZE})")
print("="*40)
print(f"ROUGE-1:        {rouge_results['rouge1']*100:.2f}%")
print(f"ROUGE-2:        {rouge_results['rouge2']*100:.2f}%")
print(f"ROUGE-L:        {rouge_results['rougeL']*100:.2f}%")
print("-" * 40)
print(f"BLEU Score:     {bleu_results['bleu']*100:.2f}")
print("-" * 40)
print(f"BERT Precision: {np.mean(bert_results['precision'])*100:.2f}%")
print(f"BERT Recall:    {np.mean(bert_results['recall'])*100:.2f}%")
print(f"BERT F1 Mean:   {np.mean(bert_results['f1'])*100:.2f}%")
print("="*40)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
⏳ Loading model from /content/drive/MyDrive/mbert...


The tokenizer you are loading from '/content/drive/MyDrive/mbert' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


✅ Model loaded on cuda
⏳ Loading dataset...
✂️ Slicing first 20 samples for evaluation...
--- Generating Predictions ---


Generating: 100%|██████████| 20/20 [00:07<00:00,  2.56it/s]


--- Calculating Metrics ---

      FINAL RESULTS (N=20)
ROUGE-1:        32.95%
ROUGE-2:        16.66%
ROUGE-L:        26.47%
----------------------------------------
BLEU Score:     4.92
----------------------------------------
BERT Precision: 74.36%
BERT Recall:    72.91%
BERT F1 Mean:   73.59%


In [15]:
print("--- Visual Inspection (First 5 Samples) ---")

for i in range(10):
    print(f"\n")
    print(f"Generated: {predictions[i]}")
    print(f"Actual:    {references[i]}")


--- Visual Inspection (First 5 Samples) ---


Generated: اسٹیبلشمنٹ کا ایک بار پھر اسٹیڈیم سے رہا کر رہا ہے ، شیخ رحمٰن
Actual:    اسٹیبلشمنٹ الیکشن سے دور رہے، قوم غنڈہ گردی برداشت نہیں کرے گی، شیخ رشید


Generated: پاک افغان طالبان نے پاک فضائیہ پر پابندیاں مسترد کردیں
Actual:    پاک افغان بارڈر پر فری موومنٹ کی اجزات، چیک پوسٹ خاتمےکےحامی عناصر کے چہرے پر ایک اور طمانچہ


Generated: ایشیا کپ : صائم ایوب نے صاؤت کو شکست دے دی ، صوبائی وزیراعظم کا انکشاف
Actual:    توجہ صرف ایک میچ پر نہیں، ایشیا کپ جیتنے پر ہے، صائم ایوب


Generated: قومی اسکیم کی شرح میں کمی ، قومی اسمبلی میں بڑھتی کمی کا اعلان
Actual:    قومی بچت اسکیمز پر شرح منافع میں کمی، مختلف اسکیمز کے نئے ریٹس جاری


Generated: پشاور میں ڈپٹی ڈائریکٹر بن گئے ہیں ، ڈونلڈ ڈار
Actual:    سلمان خان ایک غنڈہ ہے: ڈائیریکٹر ابھینیو کیشپ


Generated: خیبرپختونخوا حکومت کا سپریم کورٹ کے عہدے پر تنقید کا خدشہ
Actual:    گنڈاپور حکومت کا رویہ مضحکہ خیز ہے، پولیس اہلکاروں کی جاری تصاویر میرے سکیورٹی پرمامور اہلکار نہیں: ایمل ولی


Genera